# epic-mace — Generate TM Complexes

**epic-mace** automates stereomer search and 3D coordinate generation for mononuclear transition-metal complexes.  
This notebook walks you through the full workflow — from defining your complex to saving publication-ready XYZ files.

### Supported geometries
| Code | Geometry | Coord. number |
|------|----------|---------------|
| `OH`  | Octahedral | 6 |
| `SP`  | Square-planar | 4 |
| `TET` | Tetrahedral | 4 |
| `SPY` | Square-pyramidal | 5 |
| `TBP` | Trigonal-bipyramidal | 5 |
| `SAN` | Sandwich (two haptic centroids) | 2 |

### How to encode donor atoms
In epic-mace SMILES, **donor atoms** are tagged with atom-map numbers (`[N:1]`, `[P:2]`, …).  
Metal–ligand bonds are written as dative bonds (`->` or `<-`):
```
[Cl-:1]->[Rh+]<-[N:2]#CC
```

---

In [ ]:
# ── Install dependencies (run this cell once, then restart the kernel) ──────
# This installs epic-mace and all visualisation packages.
# If you are working from the cloned source repo, use the second command instead.

import importlib, subprocess, sys

def _install(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    else:
        print(f"{import_name} already installed")

_install('epic-mace',  'mace')
_install('rdkit',      'rdkit')
_install('py3Dmol',    'py3Dmol')
_install('ipywidgets', 'ipywidgets')

# ── If working from source (cloned repo), use this instead: ─────────────────
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
#                         'rdkit', 'py3Dmol', 'ipywidgets'])
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])

print("\nAll packages ready — you can now run the cells below.")


## 1. Setup

In [ ]:
import os
from pathlib import Path

import mace
from rdkit.Chem import Draw
import py3Dmol
from IPython.display import display

print(f"epic-mace  {mace.__version__}")

# ── helpers ────────────────────────────────────────────────────────────────

def draw2d(complexes, labels=None, size=(280, 280), mols_per_row=4):
    """Draw 2D depictions of a list of Complex objects."""
    mols = [X.mol for X in complexes]
    if labels is None:
        labels = [f"Isomer {i}" for i in range(len(mols))]
    img = Draw.MolsToGridImage(
        mols, molsPerRow=mols_per_row,
        subImgSize=size, legends=labels
    )
    display(img)


def view3d(complexes, confId=0, width=900, height=320, cols=3):
    """Show 3D structures side-by-side using py3Dmol."""
    n = len(complexes)
    cols = min(cols, n)
    rows = (n + cols - 1) // cols
    view = py3Dmol.view(
        width=width, height=height * rows,
        linked=False,
        viewergrid=(rows, cols)
    )
    for i, X in enumerate(complexes):
        r, c = divmod(i, cols)
        if X.GetNumConformers() == 0:
            continue
        view.addModel(X.ToXYZBlock(confId=confId), 'xyz', viewer=(r, c))
        view.setStyle(
            {'stick': {'radius': 0.15}, 'sphere': {'scale': 0.3}},
            viewer=(r, c)
        )
        view.setBackgroundColor('white', viewer=(r, c))
        view.zoomTo(viewer=(r, c))
    view.show()


def save_xyz(complexes, out_dir, name,
             num_repr_confs=None, e_rel_max=25.0, drop_close_energy=False):
    """Save each isomer's conformers to <out_dir>/<name>/<name>_iso#.xyz.

    Parameters
    ----------
    num_repr_confs : int or None
        Keep only this many lowest-energy conformers per isomer; None keeps all.
    e_rel_max : float
        Discard conformers above this relative energy (kJ/mol).
    drop_close_energy : bool
        Drop conformers within 0.1 kJ/mol of each other.
    """
    out = Path(out_dir) / name
    out.mkdir(parents=True, exist_ok=True)
    saved = []
    for i, X in enumerate(complexes):
        if X.GetNumConformers() == 0:
            print(f"  Isomer {i}: no conformers — skipped")
            continue
        fpath = out / f"{name}_iso{i}.xyz"
        if num_repr_confs:
            idxs = X.GetRepresentativeConfs(
                numConfs=num_repr_confs,
                dE=e_rel_max,
                dropCloseEnergy=drop_close_energy,
            )
        else:
            idxs = None
        X.ToMultipleXYZ(str(fpath), confIds=idxs)
        saved.append(str(fpath))
        print(f"  Isomer {i}: {X.GetNumConformers()} conformer(s) → {fpath.name}")
    return saved

def check_smiles(smiles, geom=None):
    """Validate a SMILES string and show a 2D preview.
    
    Checks for: parsability, donor-atom tags (:N), dative bonds (->/<-),
    and coordination number vs geometry.
    Returns True if no issues are found.
    """
    from IPython.display import display
    coord_numbers = {'OH': 6, 'SP': 4, 'TET': 4, 'SPY': 5, 'TBP': 5, 'SAN': 2}
    issues = []

    # parse
    try:
        mol = mace.MolFromSmiles(smiles)
        assert mol is not None
    except Exception:
        print("✗ Cannot parse SMILES — check brackets, charges, and bond arrows (-> / <-).")
        return False

    # donor atoms
    das = [a for a in mol.GetAtoms() if a.GetAtomMapNum() > 0]
    if len(das) == 0:
        issues.append("✗ No donor atoms found — tag them with map numbers: [N:1], [P:2], …")

    # dative bonds
    dative = [b for b in mol.GetBonds() if str(b.GetBondType()) == 'DATIVE']
    if len(dative) == 0:
        issues.append("✗ No dative bonds — connect ligands to metal with -> or <-")

    # coordination number
    if geom:
        expected = coord_numbers.get(geom.upper())
        if expected and len(das) != expected:
            issues.append(
                f"✗ {geom} needs {expected} donor atoms, "
                f"but found {len(das)} tagged atom(s)"
            )

    if issues:
        for msg in issues:
            print(msg)
        print("\n2D preview (may look odd due to issues above):")
    else:
        print(f"✓ SMILES looks good — {mol.GetNumAtoms()} atoms, "
              f"{len(das)} donor atom(s), {len(dative)} dative bond(s)")

    display(Draw.MolToImage(mol, size=(350, 250)))
    return len(issues) == 0

def save_readme(out_dir, name, stereomers,
                geom, method, complex_smiles, ligands, ca,
                regime, get_enantiomers, min_trans_cycle, mer_rule,
                num_confs, rms_thresh,
                num_repr_confs, e_rel_max, drop_close_energy,
                extra_notes=''):
    """Write README.md into <out_dir>/<name>/ documenting this run.

    Call after save_xyz() so the file list is complete.
    extra_notes: any free-text observations to append (next steps, comments, …).
    """
    import datetime
    geom_names = {
        'OH': 'Octahedral (OH)',   'SP': 'Square-planar (SP)',
        'TET': 'Tetrahedral (TET)', 'SPY': 'Square-pyramidal (SPY)',
        'TBP': 'Trigonal-bipyramidal (TBP)', 'SAN': 'Sandwich (SAN)',
    }
    ts  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    out = Path(out_dir) / name
    out.mkdir(parents=True, exist_ok=True)

    L = []
    L += [f'# epic-mace run: {name}', '',
          f'**Generated:** {ts}  ',
          f'**epic-mace version:** {mace.__version__}',
          '', '---', '## 1. Input', '',
          '| Parameter | Value |', '|-----------|-------|',
          f'| Name | `{name}` |',
          f'| Geometry | {geom_names.get(geom, geom)} |',
          f'| Input method | {method} |']

    if method == 'smiles':
        L.append(f'| Complex SMILES | `{complex_smiles}` |')
    else:
        L.append(f'| Central atom | `{ca}` |')
        L += ['', '**Ligands:**', '']
        for lig in (ligands or []):
            L.append(f'- `{lig}`')

    L += ['', '---', '## 2. Stereomer search', '',
          '| Parameter | Value |', '|-----------|-------|',
          f'| Regime | `{regime}` |',
          f'| Include enantiomers | `{get_enantiomers}` |',
          f'| Min trans cycle | `{min_trans_cycle}` |',
          f'| Mer rule | `{mer_rule}` |',
          '', '---', '## 3. 3D structure generation', '',
          '| Parameter | Value |', '|-----------|-------|',
          f'| Conformers attempted | `{num_confs}` |',
          f'| RMSD threshold | `{rms_thresh}` Å |',
          f'| Representative conformers kept | `{num_repr_confs if num_repr_confs else "all"}` |',
          f'| Max relative energy | `{e_rel_max}` kJ/mol |',
          f'| Drop near-degenerate conformers | `{drop_close_energy}` |',
          '', '---', '## 4. Results', '']

    if any(X.GetNumConformers() > 0 for X in stereomers):
        L += ['| Isomer | File | Conformers | Lowest E (kJ/mol) | SMILES |',
              '|--------|------|:----------:|:-----------------:|--------|']
        for i, X in enumerate(stereomers):
            n   = X.GetNumConformers()
            fn  = f'{name}_iso{i}.xyz' if n else '*(no conformers)*'
            e   = f'{X.GetConfEnergy(0):.1f}' if n else '—'
            smi = mace.MolToSmiles(X.mol)
            smi = smi[:70] + '…' if len(smi) > 70 else smi
            L.append(f'| {i} | `{fn}` | {n} | {e} | `{smi}` |')
    else:
        L.append('*No conformers were generated.*')

    n_tot  = sum(X.GetNumConformers() for X in stereomers)
    n_fail = sum(1 for X in stereomers if X.GetNumConformers() == 0)
    note   = f', {n_fail} isomer(s) had no conformers' if n_fail else ''
    L += ['',
          f'**Summary:** {len(stereomers)} stereoisomer(s) found, '
          f'{n_tot} conformer(s) saved{note}.']

    if extra_notes:
        L += ['', '---', '## Notes', '', extra_notes]

    p = out / 'README.md'
    p.write_text('\n'.join(L))
    print(f'README written → {p}')
    return str(p)


---
## 2. Define your complex

Choose **one** of the two input methods below, and comment out the other.

### Method A — full complex SMILES  
Write the entire complex as a single SMILES string with dative bonds (`->` / `<-`).

### Method B — separate ligands + central atom  
List each ligand as its own SMILES and specify the metal separately; epic-mace assembles the complex.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  USER INPUT — edit values below, then run the cell
# ══════════════════════════════════════════════════════════════════

# Geometry of the central atom  (OH | SP | TET | SPY | TBP | SAN)
GEOM = 'OH'

# Friendly name used for output folders/files
NAME = 'RuSNS_H2_PPh3'

# ── Choose input method ───────────────────────────────────────────
# Set METHOD = 'smiles'   to define the whole complex as one SMILES string
# Set METHOD = 'ligands'  to list ligands + central atom separately
METHOD = 'ligands'

# ── Method A : full complex SMILES ───────────────────────────────
# (only used when METHOD = 'smiles')
COMPLEX_SMILES = '[NH3:1]->[Pt+2](<-[NH3:1])(<-[Cl-:1])(<-[Cl-:1])'

# ── Method B : separate ligands + central atom ───────────────────
# (only used when METHOD = 'ligands')
#
# RuSNS_H2_PPh3: Ru(II) octahedral, SNS pincer + PPh3 + 2 hydrides
# Donor atoms tagged with map numbers:
#   [S:1]  first sulfur   [NH:2]  amine nitrogen (H explicit!)   [S:3]  second sulfur
#   [P:4]  phosphorus     [H-:5] hydride (x2, same tag = same ligand type)
LIGANDS = [
    'CC[S:1]CC[NH:2]CC[S:3]CC',                   # SNS pincer (tridentate)
    '[P:4](c1ccccc1)(c1ccccc1)c1ccccc1',          # PPh3
    '[H-:5]',                                      # hydride 1
    '[H-:5]',                                      # hydride 2
]
CA = '[Ru+2]'

# Other examples:
#   Cisplatin, square-planar  (set GEOM='SP', METHOD='smiles'):
#     COMPLEX_SMILES = '[NH3:1]->[Pt+2](<-[NH3:1])(<-[Cl-:1])(<-[Cl-:1])'
#   [Co(NH3)4Cl2]+, octahedral  (set GEOM='OH', METHOD='ligands'):
#     LIGANDS = ['[NH3:1]','[NH3:1]','[NH3:1]','[NH3:1]','[Cl-:2]','[Cl-:2]']
#     CA = '[Co+3]'

# ══════════════════════════════════════════════════════════════════

# build the Complex object
if METHOD == 'smiles':
    X0 = mace.Complex(smiles=COMPLEX_SMILES, geom=GEOM)
    print(f"Complex loaded from SMILES  |  geometry: {GEOM}")
elif METHOD == 'ligands':
    X0 = mace.ComplexFromLigands(LIGANDS, CA, GEOM)
    print(f"Complex assembled from ligands  |  geometry: {GEOM}")
else:
    raise ValueError(f"Unknown METHOD={METHOD!r}. Use 'smiles' or 'ligands'.")

print(f"Atoms: {X0.mol.GetNumAtoms()}   Donor atoms: {len(X0._DAs)}")
if X0.err_init:
    print(f"Note: {X0.err_init}")
    print("      → Run the Stereomer search cell to resolve the geometry.")
else:
    draw2d([X0], labels=[NAME])


In [ ]:
# ── Quick SMILES check ───────────────────────────────────────────────────
# Run this any time you change COMPLEX_SMILES to catch mistakes early.
# (When using METHOD='ligands', mace assembles the SMILES automatically —
#  you can inspect the assembled result below instead.)

if METHOD == 'smiles':
    check_smiles(COMPLEX_SMILES, geom=GEOM)
else:
    assembled = mace.MolToSmiles(X0.mol)
    print(f"Assembled SMILES: {assembled}")
    print(f"Atoms: {X0.mol.GetNumAtoms()}   Donor atoms: {len(X0._DAs)}")
    from rdkit.Chem import Draw
    from IPython.display import display
    display(Draw.MolToImage(X0.mol, size=(400, 280)))


---
## 3. Stereomer search

| `regime` | What is iterated |
|----------|------------------|
| `'all'`     | All stereocenters (central atom **and** ligands) |
| `'CA'`      | Central-atom configuration only |
| `'ligands'` | Ligand stereocenters only |
| `'none'`    | No search — use the input structure as-is |

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  USER INPUT
# ══════════════════════════════════════════════════════════════════

REGIME            = 'all'   # 'all' | 'CA' | 'ligands' | 'none'
GET_ENANTIOMERS   = False   # True → keep both enantiomers for chiral complexes
MIN_TRANS_CYCLE   = None    # int or None: min bonds needed for a trans-DA arrangement
MER_RULE          = True    # True → forbid fac- for rigid DA-DA-DA fragments

# ══════════════════════════════════════════════════════════════════

if REGIME == 'none':
    stereomers = [X0]
else:
    stereomers = X0.GetStereomers(
        regime          = REGIME,
        dropEnantiomers = not GET_ENANTIOMERS,
        minTransCycle   = MIN_TRANS_CYCLE,
        merRule         = MER_RULE,
    )

print(f"Found {len(stereomers)} stereomer(s)")
draw2d(stereomers, labels=[f"Isomer {i}" for i in range(len(stereomers))])


---
## 4. Generate 3D structures

| Parameter | Meaning |
|-----------|--------|
| `NUM_CONFS` | Conformers attempted per isomer. More = better coverage, slower. |
| `RMS_THRESH` | Drop a conformer if its RMSD to any kept conformer is < this (Å). `−1` = keep all. |
| `NUM_REPR_CONFS` | After generation, retain only the N best conformers (`None` = all). |
| `E_REL_MAX` | Discard conformers with relative energy > this value (kJ/mol). |
| `DROP_CLOSE_ENERGY` | Drop conformers within 0.1 kJ/mol of each other. |

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  USER INPUT
# ══════════════════════════════════════════════════════════════════

NUM_CONFS         = 10    # conformers to attempt per isomer
RMS_THRESH        = 0.5   # Å; -1 = keep all (no RMSD filtering)

NUM_REPR_CONFS    = 1     # keep N lowest-energy conformers per isomer; None = all
E_REL_MAX         = 25.0  # kJ/mol energy window above minimum
DROP_CLOSE_ENERGY = True  # drop near-degenerate conformers (ΔE < 0.1 kJ/mol)

# ══════════════════════════════════════════════════════════════════

for i, X in enumerate(stereomers):
    X.AddConformers(numConfs=NUM_CONFS, rmsThresh=RMS_THRESH)
    X.OrderConfsByEnergy()
    n = X.GetNumConformers()
    if n:
        E = round(X.GetConfEnergy(0), 2)  # conf 0 = lowest after OrderConfsByEnergy
        print(f"  Isomer {i}: {n:3d} conformer(s)  |  lowest E = {E} kJ/mol")
    else:
        print(f"  Isomer {i}: *** embedding failed — 0 conformers ***")


---
## 5. Visualise 3D structures

In [ ]:
# Show the lowest-energy conformer for every isomer (side-by-side)
has_confs = [X for X in stereomers if X.GetNumConformers() > 0]

if not has_confs:
    print("No conformers to display.")
else:
    view3d(has_confs, confId=0)

In [ ]:
# Single-isomer large view  (change ISO_IDX to inspect a different isomer)
ISO_IDX = 0

X = stereomers[ISO_IDX]
if X.GetNumConformers() == 0:
    print("No conformers for this isomer.")
else:
    view = py3Dmol.view(width=480, height=400)
    view.addModel(X.ToXYZBlock(confId=0), 'xyz')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.35}})
    view.setBackgroundColor('white')
    view.zoomTo()
    view.show()

---
## 6. Save XYZ files

Files are written to `<OUT_DIR>/<NAME>/<NAME>_iso#.xyz`.  
Multi-conformer isomers produce a multi-frame XYZ file (one frame per conformer).

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  USER INPUT
# ══════════════════════════════════════════════════════════════════

OUT_DIR = './output'   # root directory (created if absent)

# ══════════════════════════════════════════════════════════════════

saved = save_xyz(
    stereomers, OUT_DIR, NAME,
    num_repr_confs    = NUM_REPR_CONFS,
    e_rel_max         = E_REL_MAX,
    drop_close_energy = DROP_CLOSE_ENERGY,
)
print(f"\nAll files written to:  {(Path(OUT_DIR) / NAME).resolve()}")

---
## 7. Save README

Writes a `README.md` into the same output folder as the XYZ files,
documenting every input parameter and the results table.
Add any free-text observations in `NOTES` (leave blank to omit).

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  USER INPUT (optional)
# ══════════════════════════════════════════════════════════════════

# Free-text notes appended as a Notes section (leave empty to omit)
NOTES = ""

# ══════════════════════════════════════════════════════════════════

save_readme(
    out_dir           = OUT_DIR,
    name              = NAME,
    stereomers        = stereomers,
    # ── complex definition ──────────────────────────────────────
    geom              = GEOM,
    method            = METHOD,
    complex_smiles    = COMPLEX_SMILES if METHOD == 'smiles' else None,
    ligands           = LIGANDS        if METHOD == 'ligands' else None,
    ca                = CA             if METHOD == 'ligands' else None,
    # ── stereomer search ────────────────────────────────────────
    regime            = REGIME,
    get_enantiomers   = GET_ENANTIOMERS,
    min_trans_cycle   = MIN_TRANS_CYCLE,
    mer_rule          = MER_RULE,
    # ── conformer generation ────────────────────────────────────
    num_confs         = NUM_CONFS,
    rms_thresh        = RMS_THRESH,
    num_repr_confs    = NUM_REPR_CONFS,
    e_rel_max         = E_REL_MAX,
    drop_close_energy = DROP_CLOSE_ENERGY,
    # ── optional notes ──────────────────────────────────────────
    extra_notes       = NOTES,
)


---
## 8. (Advanced) Substituent screening

Replace dummy-atom placeholders in the SMILES (`[1*]`, `[2*]`, …) with real groups  
and generate all combinations automatically.

Encode attachment points with isotopically-labelled dummy atoms:  
- `[1*]` → replaced by **R1**  
- `[2*]` → replaced by **R2**  
- … etc.

Each substituent SMILES must have exactly **one** dummy atom `[*]` at the attachment site.

In [ ]:
from itertools import product as iterproduct

# ══════════════════════════════════════════════════════════════════
#  USER INPUT
# ══════════════════════════════════════════════════════════════════

# Core SMILES — ligand with [1*]/[2*] placeholders.
# Example: bipyridine ligand with two R-positions
CORE_SMILES  = "[1*]C1=C[N:4]=C(C=C1)C1=[N:3]C=C([2*])C=C1"
SCREEN_GEOM  = 'SP'
SCREEN_NAME  = 'bipy_screening'
SCREEN_OUT   = './output'
SCREEN_REGIME = 'none'   # usually 'none' or 'CA'

# Substituent libraries  —  name → attachment SMILES  ([*] = bond site)
R1_subs = {
    'H':    '[*][H]',
    'Me':   '[*]C',
    'OMe':  '[*]OC',
    'NMe2': '[*]N(C)C',
    'F':    '[*]F',
    'CF3':  '[*]C(F)(F)F',
}
R2_subs = {
    'H':    '[*][H]',
    'Me':   '[*]C',
    'OMe':  '[*]OC',
}

SCREEN_NUM_CONFS   = 5
SCREEN_RMS_THRESH  = 0.5
SCREEN_REPR_CONFS  = 1

# ══════════════════════════════════════════════════════════════════

core_mol = mace.MolFromSmiles(CORE_SMILES)
results  = {}   # fullname → list[Complex]

for (r1_name, r1_smi), (r2_name, r2_smi) in iterproduct(R1_subs.items(), R2_subs.items()):
    fullname = f"{SCREEN_NAME}_R1-{r1_name}_R2-{r2_name}"
    subs = {
        'R1': mace.MolFromSmiles(r1_smi),
        'R2': mace.MolFromSmiles(r2_smi),
    }
    mol = mace.AddSubsToMol(core_mol, subs)
    X   = mace.ComplexFromMol(mol, SCREEN_GEOM)

    if SCREEN_REGIME == 'none':
        Xs = [X]
    else:
        Xs = X.GetStereomers(regime=SCREEN_REGIME, dropEnantiomers=True)

    for Xi in Xs:
        Xi.AddConformers(numConfs=SCREEN_NUM_CONFS, rmsThresh=SCREEN_RMS_THRESH)
        Xi.OrderConfsByEnergy()

    results[fullname] = Xs
    n_confs = sum(Xi.GetNumConformers() for Xi in Xs)
    print(f"  {fullname}: {len(Xs)} isomer(s), {n_confs} conformer(s)")

print(f"\nScreening complete — {len(results)} systems")

In [ ]:
# Save all screening results
for fullname, Xs in results.items():
    save_xyz(
        Xs, SCREEN_OUT, fullname,
        num_repr_confs=SCREEN_REPR_CONFS, e_rel_max=25.0, drop_close_energy=True
    )

print(f"\nAll screening files written to: {Path(SCREEN_OUT).resolve()}")

---
## 9. (Advanced) Run from an existing YAML file

If you already have an epic-mace YAML input file (from the CLI workflow),  
run it directly here without leaving the notebook.

In [ ]:
from mace.__main__ import (
    get_args_from_file, args_to_command,
    check_arguments, prepare_complexes, run_mace_for_system,
    get_parser, read_subs,
)

# ══════════════════════════════════════════════════════════════════
#  USER INPUT
# ══════════════════════════════════════════════════════════════════

YAML_INPUT = './examples/01_complex/input_complex.yaml'

# ══════════════════════════════════════════════════════════════════

raw        = get_args_from_file(YAML_INPUT)
cmd        = args_to_command(raw)
parser     = get_parser()
args, unk  = parser.parse_known_args(cmd)
subs_args  = read_subs(unk)
all_args   = {**args.__dict__, **subs_args.__dict__}

params = check_arguments(all_args)
jobs   = prepare_complexes(params)

for fullname, X in jobs.items():
    run_mace_for_system(X, fullname, params)
    print(f"Done: {fullname}")

---
## Appendix — common complex SMILES

Copy-paste any block into Section 2 to get started quickly.

```python
# ── Square-planar ─────────────────────────────────────────────────

# Cisplatin  [Pt(NH3)2Cl2]
GEOM = 'SP'; NAME = 'cisplatin'
COMPLEX_SMILES = '[NH3:1]->[Pt+2](<-[NH3:1])(<-[Cl-:1])(<-[Cl-:1])'

# Rh(bipy)(MeCN)Cl
GEOM = 'SP'; NAME = 'Rh_bipy_MeCN_Cl'
COMPLEX_SMILES = '[Cl-:1]->[Rh+]<-1(<-[N:2]#CC)<-[N:3]2=CC=CC=C2C2=[N:4]1C=CC=C2'

# ── Octahedral ────────────────────────────────────────────────────

# [Co(NH3)4Cl2]+  — fac/mer isomers
GEOM = 'OH'; NAME = 'Co_NH3_Cl'
LIGANDS = ['[NH3:1]', '[NH3:1]', '[NH3:1]', '[NH3:1]', '[Cl-:1]', '[Cl-:1]']
CA = '[Co+3]'

# Mn(PNP)(CO)3Br  — mer-tridentate PNP ligand
GEOM = 'OH'; NAME = 'Mn_PNP_CO3_Br'
LIGANDS = [
    '[P:1](C)(C)CC[N:2](CC[P:3](C)C)',   # PNP tridentate
    '[C:4]#[O]', '[C:5]#[O]', '[C:6]#[O]',
    '[Br-:7]',
]
CA = '[Mn+]'

# ── Tetrahedral ───────────────────────────────────────────────────

# [ZnCl2(py)2]
GEOM = 'TET'; NAME = 'Zn_Cl2_py2'
LIGANDS = ['[Cl-:1]', '[Cl-:1]', '[N:2]1=CC=CC=C1', '[N:2]1=CC=CC=C1']
CA = '[Zn+2]'

# ── Trigonal-bipyramidal ──────────────────────────────────────────

# [Fe(CO)5]
GEOM = 'TBP'; NAME = 'Fe_CO5'
LIGANDS = ['[C:1]#[O]', '[C:1]#[O]', '[C:1]#[O]', '[C:1]#[O]', '[C:1]#[O]']
CA = '[Fe]'
```